# Task 3: Evaluation, Optimisation and Explainability
**7006SCN Machine Learning and Big Data** | Week 5

This notebook (1) reads Spark's stage timings, (2) makes two measured speed-ups with before and after timings, (3) tests how stable the four Task 2 models are when the inputs change a little, and (4) explains the logistic regression with SHAP and checks one bias risk.

**How to run:** run all cells from the top in one session. The four models are loaded from `models/` if Task 2 left them on disk; otherwise they are refitted with the settings in the `BEST` cell (copy the best parameters printed at the end of Task 2 into that cell).

## 1. Declaration

In [ ]:
# --- Student declaration ---
STUDENT_NAME  = "Sridhar Gude"
STUDENT_EMAIL = "gudes@coventry.ac.uk"
SID           = "17165640"
POOL_REF      = "No. 33 (Dataset Catalogue)"
DATASET_NAME  = "NYC TLC Yellow Taxi Trip Records, Jan-Jun 2024"
HEADER = f"{STUDENT_NAME} | {STUDENT_EMAIL} | {DATASET_NAME} | Pool ref {POOL_REF}"
print(HEADER)

## 2. Setup

In [ ]:
# ---------- Setup: load the Task 1 data if it exists, otherwise rebuild it ----------
import os, sys, json, time, glob, urllib.request, subprocess
from functools import reduce
try:
    import pyspark
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "pyspark"])
from pyspark.sql import SparkSession, DataFrame, functions as F
from pyspark import StorageLevel
from pyspark.ml import Pipeline, PipelineModel
from pyspark.ml.feature import (SQLTransformer, Imputer, VectorAssembler, StandardScaler,
                                StringIndexer, OneHotEncoder)

RAW_DIR, OUT_DIR, MODEL_DIR, RES_DIR = "data/raw", "data/processed", "models", "results"
for d in (RAW_DIR, OUT_DIR, MODEL_DIR, RES_DIR):
    os.makedirs(d, exist_ok=True)
timings = {}

spark = (SparkSession.builder.appName("7006SCN_Task_NYC_Taxi_Payment").master("local[*]")
         .config("spark.driver.memory", "8g")
         .config("spark.sql.shuffle.partitions", "64")
         .config("spark.sql.session.timeZone", "UTC").getOrCreate())
sc = spark.sparkContext
CLUSTER = {"spark_version": spark.version, "master": sc.master, "cores": sc.defaultParallelism,
           "driver_memory": spark.conf.get("spark.driver.memory", "default"),
           "shuffle_partitions": spark.conf.get("spark.sql.shuffle.partitions"),
           "note": "local mode: one JVM, the driver also runs the tasks (no separate executors)"}
print(CLUSTER)

BASE  = "https://d37ci6vzurychx.cloudfront.net/trip-data/"
FILES = [f"yellow_tripdata_2024-{m:02d}.parquet" for m in range(1, 7)]
COMMON = {"vendorid": "int", "tpep_pickup_datetime": "timestamp", "tpep_dropoff_datetime": "timestamp",
          "passenger_count": "double", "trip_distance": "double", "ratecodeid": "double",
          "store_and_fwd_flag": "string", "pulocationid": "int", "dolocationid": "int",
          "payment_type": "int", "fare_amount": "double", "extra": "double", "mta_tax": "double",
          "tip_amount": "double", "tolls_amount": "double", "improvement_surcharge": "double",
          "total_amount": "double", "congestion_surcharge": "double", "airport_fee": "double"}

def load_raw():
    parts = []
    for f in FILES:
        path = os.path.join(RAW_DIR, f)
        if not os.path.exists(path):
            print("downloading", f)
            urllib.request.urlretrieve(BASE + f, path)
        d = spark.read.parquet(path)
        d = d.toDF(*[c.lower() for c in d.columns])
        parts.append(d.select([F.col(c).cast(t).alias(c) for c, t in COMMON.items()]))
    return reduce(DataFrame.unionByName, parts)

feature_sql = """
SELECT *,
  (unix_timestamp(tpep_dropoff_datetime) - unix_timestamp(tpep_pickup_datetime)) / 60.0 AS trip_minutes,
  trip_distance / ((unix_timestamp(tpep_dropoff_datetime) - unix_timestamp(tpep_pickup_datetime)) / 3600.0) AS avg_speed_mph,
  fare_amount / trip_distance AS fare_per_mile,
  hour(tpep_pickup_datetime) AS pickup_hour,
  dayofweek(tpep_pickup_datetime) AS pickup_dow,
  CASE WHEN dayofweek(tpep_pickup_datetime) IN (1, 7) THEN 1.0 ELSE 0.0 END AS is_weekend,
  CASE WHEN passenger_count IS NULL THEN 1.0 ELSE 0.0 END AS passenger_missing,
  CASE WHEN congestion_surcharge IS NULL THEN 1.0 ELSE 0.0 END AS congestion_missing,
  CASE WHEN airport_fee IS NULL THEN 1.0 ELSE 0.0 END AS airport_missing,
  CAST(vendorid AS STRING) AS vendor_c,
  COALESCE(CAST(CAST(ratecodeid AS INT) AS STRING), 'unknown') AS rate_c,
  COALESCE(store_and_fwd_flag, 'unknown') AS flag_c,
  CAST(pulocationid AS STRING) AS pu_c,
  CAST(dolocationid AS STRING) AS do_c
FROM __THIS__
"""
to_impute = ["passenger_count", "congestion_surcharge", "airport_fee"]
num_cols  = ["passenger_count_imp", "congestion_surcharge_imp", "airport_fee_imp", "trip_distance",
             "fare_amount", "extra", "mta_tax", "tolls_amount", "improvement_surcharge", "trip_minutes",
             "avg_speed_mph", "fare_per_mile", "pickup_hour", "pickup_dow"]
flag_cols = ["is_weekend", "passenger_missing", "congestion_missing", "airport_missing"]
cat_cols  = ["vendor_c", "rate_c", "flag_c", "pu_c", "do_c"]

def build_pipeline():
    return Pipeline(stages=[
        SQLTransformer(statement=feature_sql),
        Imputer(strategy="median", inputCols=to_impute, outputCols=[c + "_imp" for c in to_impute]),
        VectorAssembler(inputCols=num_cols, outputCol="num_vec"),
        StandardScaler(inputCol="num_vec", outputCol="num_scaled", withMean=False, withStd=True),
        StringIndexer(inputCols=cat_cols, outputCols=[c + "_idx" for c in cat_cols], handleInvalid="keep"),
        OneHotEncoder(inputCols=[c + "_idx" for c in cat_cols], outputCols=[c + "_ohe" for c in cat_cols],
                      handleInvalid="keep"),
        VectorAssembler(inputCols=["num_scaled"] + [c + "_ohe" for c in cat_cols] + flag_cols,
                        outputCol="features")])

def clean_rows(raw):
    pick, drop_ = F.col("tpep_pickup_datetime"), F.col("tpep_dropoff_datetime")
    minutes = (F.unix_timestamp(drop_) - F.unix_timestamp(pick)) / 60.0
    rules = [(pick >= F.to_timestamp(F.lit("2024-01-01"))) & (pick < F.to_timestamp(F.lit("2024-07-01"))),
             (minutes >= 1) & (minutes <= 180),
             (F.col("trip_distance") >= 0.1) & (F.col("trip_distance") <= 100),
             (F.col("fare_amount") > 0) & (F.col("fare_amount") <= 500),
             F.col("payment_type").isin(1, 2)]
    keep = reduce(lambda a, b: a & b, [F.coalesce(r, F.lit(False)) for r in rules])
    return (raw.filter(keep)
               .withColumn("is_cash", (F.col("payment_type") == 2).cast("int"))
               .withColumn("passenger_count", F.when(F.col("passenger_count").between(1, 6), F.col("passenger_count")))
               .fillna(0, subset=["extra", "mta_tax", "tolls_amount", "improvement_surcharge"])
               .drop("tip_amount", "total_amount", "payment_type"))

train_path, test_path = os.path.join(OUT_DIR, "train.parquet"), os.path.join(OUT_DIR, "test.parquet")
model_path = os.path.join(MODEL_DIR, "prep_pipeline")
if os.path.exists(train_path) and os.path.exists(test_path) and os.path.exists(model_path):
    print("Found Task 1 data and pipeline on disk: loading them")
    train, test = spark.read.parquet(train_path), spark.read.parquet(test_path)
    prep_model = PipelineModel.load(model_path)
else:
    print("Task 1 files not found: rebuilding them (takes a few minutes)")
    t0 = time.time()
    clean = clean_rows(load_raw()).persist(StorageLevel.MEMORY_AND_DISK)
    train, test = clean.randomSplit([0.8, 0.2], seed=42)
    prep_model = build_pipeline().fit(train)
    train.write.mode("overwrite").parquet(train_path)
    test.write.mode("overwrite").parquet(test_path)
    prep_model.write().overwrite().save(model_path)
    train, test = spark.read.parquet(train_path), spark.read.parquet(test_path)
    print("rebuilt in", round(time.time() - t0, 1), "s")

## 3. Modelling data (same samples and seeds as Task 2)
`train_p` is deliberately **not** cached yet, so the first experiment can time it before and after caching.

In [ ]:
import numpy as np, pandas as pd
from pyspark.ml.functions import vector_to_array
from pyspark.ml.classification import (LogisticRegression, LogisticRegressionModel,
    RandomForestClassifier, RandomForestClassificationModel, GBTClassifier, GBTClassificationModel,
    MultilayerPerceptronClassifier, MultilayerPerceptronClassificationModel)
from pyspark.ml.evaluation import BinaryClassificationEvaluator

TRAIN_FRACTION, TEST_FRACTION = 0.03, 0.30
train_s = train.sampleBy("is_cash", fractions={0: TRAIN_FRACTION, 1: TRAIN_FRACTION}, seed=42)
test_s  = test.sample(False, TEST_FRACTION, seed=7)

def to_model_frame(df):
    return (prep_model.transform(df)
              .select(F.col("is_cash").cast("double").alias("label"), "features", "pu_c"))

train_p = to_model_frame(train_s)                                  # NOT cached yet
test_p  = to_model_frame(test_s).persist(StorageLevel.MEMORY_AND_DISK)
n_test  = test_p.count()
n_features = test_p.head()["features"].size
print(f"test rows: {n_test:,}   features: {n_features}")

## 4. Optimisation 1: cache the training frame
`train_p` is rebuilt from the Parquet files and the pipeline every time a model reads it. Fitting a model reads it many times (one pass per iteration), so I expect caching to help. I time the same logistic regression fit before and after `persist`.

In [ ]:
probe = LogisticRegression(maxIter=20, regParam=0.01, standardization=False)

t0 = time.time(); probe.fit(train_p); t_before = round(time.time() - t0, 1)

train_c = to_model_frame(train_s).persist(StorageLevel.MEMORY_AND_DISK)
t0 = time.time(); n_train = train_c.count(); t_build = round(time.time() - t0, 1)
t0 = time.time(); probe.fit(train_c); t_after = round(time.time() - t0, 1)

print(f"train rows: {n_train:,}")
print(f"fit WITHOUT cache : {t_before} s")
print(f"cost to fill cache: {t_build} s (one-off)")
print(f"fit WITH cache    : {t_after} s")
opt_cache = {"change": "persist training frame (MEMORY_AND_DISK)", "setting": "persist before fit",
             "before_s": t_before, "after_s": t_after, "one_off_cache_fill_s": t_build}

## 5. Optimisation 2: shuffle partitions
A `groupBy` shuffles data into `spark.sql.shuffle.partitions` pieces. The default of 200 is far more than this 4-core machine needs, and each tiny task has a fixed cost. I time the same aggregation (trips and cash rate per pickup zone) with 200 and with 16 partitions. Adaptive execution is switched off for this test so the setting is not changed behind my back.

In [ ]:
def shuffle_time(parts):
    spark.conf.set("spark.sql.adaptive.enabled", "false")
    spark.conf.set("spark.sql.shuffle.partitions", str(parts))
    t0 = time.time()
    test_p.groupBy("pu_c").agg(F.count("*").alias("trips"), F.avg("label").alias("cash_rate")).collect()
    return round(time.time() - t0, 2)

shuffle_time(200)                      # warm-up, not reported
runs_200 = [shuffle_time(200) for _ in range(3)]
runs_16  = [shuffle_time(16) for _ in range(3)]
t_200, t_16 = round(float(np.median(runs_200)), 2), round(float(np.median(runs_16)), 2)

spark.conf.set("spark.sql.adaptive.enabled", "true")
spark.conf.set("spark.sql.shuffle.partitions", "64")
print(f"200 shuffle partitions: {runs_200} -> median {t_200} s")
print(f" 16 shuffle partitions: {runs_16} -> median {t_16} s")
opt_shuffle = {"change": "shuffle partitions", "setting": "spark.sql.shuffle.partitions 200 -> 16",
               "before_s": t_200, "after_s": t_16}

## 6. Reading the Spark UI
Spark's web UI is not reachable from this notebook, so I read the same numbers from its REST API: stage run time, shuffle read and write, spill, and the task time spread (max against median) for the three slowest stages. A max-to-median ratio far above 1 means a few tasks are much slower than the rest (skew).

**Fig. T3.1**: stages ranked by run time.

In [ ]:
import urllib.request, matplotlib.pyplot as plt

def get_json(url):
    with urllib.request.urlopen(url, timeout=20) as r:
        return json.load(r)

ui_candidates = [(sc.uiWebUrl or "").rstrip("/"), "http://localhost:4040"]
stages, ui_base, app_id = None, None, None
for base in [b for b in ui_candidates if b]:
    try:
        app_id = get_json(base + "/api/v1/applications")[0]["id"]
        stages = get_json(f"{base}/api/v1/applications/{app_id}/stages?status=complete")
        ui_base = base; break
    except Exception as e:
        print("could not reach", base, "->", e)

stage_table = pd.DataFrame()
if stages:
    stage_table = pd.DataFrame([{
        "stage": s["stageId"], "name": s["name"][:48], "tasks": s["numTasks"],
        "run_s": round(s["executorRunTime"] / 1000, 1),
        "shuffle_read_MB": round(s.get("shuffleReadBytes", 0) / 1e6, 1),
        "shuffle_write_MB": round(s.get("shuffleWriteBytes", 0) / 1e6, 1),
        "spill_MB": round((s.get("memoryBytesSpilled", 0) + s.get("diskBytesSpilled", 0)) / 1e6, 1),
        "attempt": s["attemptId"]} for s in stages]).sort_values("run_s", ascending=False)
    top = stage_table.head(10).copy()

    skew = []
    for _, r in top.head(3).iterrows():
        try:
            q = get_json(f"{ui_base}/api/v1/applications/{app_id}/stages/{int(r['stage'])}/{int(r['attempt'])}"
                         f"/taskSummary?quantiles=0.5,1.0")["executorRunTime"]
            skew.append(round(q[1] / q[0], 1) if q[0] else None)
        except Exception:
            skew.append(None)
    top["task_max_to_median"] = skew + [None] * (len(top) - len(skew))
    print(top.drop(columns="attempt").to_string(index=False))

    fig, ax = plt.subplots(figsize=(9, 4))
    ax.barh(top["name"].str.slice(0, 40) + "  (stage " + top["stage"].astype(str) + ")", top["run_s"])
    ax.invert_yaxis(); ax.set_xlabel("executor run time (s)"); ax.set_title("Spark stages by run time")
    plt.tight_layout(); plt.show()
else:
    print("Spark UI API not reachable. Use a screenshot of the Spark UI Stages tab instead if you can open it.")

## 7. Load or refit the four models
If the Task 2 models are on disk they are loaded. Otherwise they are refitted with the settings below (edit them to match the best parameters from Task 2).

In [ ]:
BEST = {"LogisticRegression": {"regParam": 0.01}, "RandomForest": {"maxDepth": 10},
        "GBTClassifier": {"maxDepth": 5}, "MultilayerPerceptron": {"hidden": 32}}

loaders = {"LogisticRegression": LogisticRegressionModel, "RandomForest": RandomForestClassificationModel,
           "GBTClassifier": GBTClassificationModel, "MultilayerPerceptron": MultilayerPerceptronClassificationModel}

def make_estimator(name):
    if name == "LogisticRegression":
        return LogisticRegression(maxIter=40, standardization=False, regParam=BEST[name]["regParam"])
    if name == "RandomForest":
        return RandomForestClassifier(numTrees=30, maxBins=32, seed=42, maxDepth=BEST[name]["maxDepth"])
    if name == "GBTClassifier":
        return GBTClassifier(maxIter=20, seed=42, maxDepth=BEST[name]["maxDepth"])
    return MultilayerPerceptronClassifier(maxIter=40, blockSize=128, seed=42,
                                          layers=[n_features, BEST[name]["hidden"], 2])

models = {}
for name in loaders:
    path = os.path.join(MODEL_DIR, name)
    if os.path.exists(path):
        models[name] = loaders[name].load(path); how = "loaded from Task 2"
    else:
        models[name] = make_estimator(name).fit(train_c); how = "refitted with BEST settings"
    print(f"{name:22s} {how}")

## 8. Stability: perturb the inputs
I change `fare_amount` and `trip_distance` of every test trip by a random amount of up to plus or minus 5%, then re-run the full Task 1 pipeline (so speed and fare per mile change too) and score the same models again. A stable model loses little. The rank orders the models by the average size of the three changes (1 = most stable).

In [ ]:
PERT = 0.05
def perturb(df):
    return (df.withColumn("fare_amount", F.col("fare_amount") * (1 + PERT * (2 * F.rand(11) - 1)))
              .withColumn("trip_distance", F.col("trip_distance") * (1 + PERT * (2 * F.rand(12) - 1))))

test_pert_p = to_model_frame(perturb(test_s)).persist(StorageLevel.MEMORY_AND_DISK)
test_pert_p.count()

roc_eval, pr_eval = BinaryClassificationEvaluator(metricName="areaUnderROC"), BinaryClassificationEvaluator(metricName="areaUnderPR")

def score(model, frame):
    p = model.transform(frame).select("label", "prediction", "rawPrediction").persist()
    roc, pr = roc_eval.evaluate(p), pr_eval.evaluate(p)
    cm = {(int(r["label"]), int(r["prediction"])): int(r["count"]) for r in p.groupBy("label", "prediction").count().collect()}
    tp, fp, fn, tn = cm.get((1, 1), 0), cm.get((0, 1), 0), cm.get((1, 0), 0), cm.get((0, 0), 0)
    prec = tp / (tp + fp) if (tp + fp) else 0.0
    rec = tp / (tp + fn) if (tp + fn) else 0.0
    f1 = 2 * prec * rec / (prec + rec) if (prec + rec) else 0.0
    p.unpersist()
    return {"f1_cash": f1, "roc_auc": roc, "pr_auc": pr}

rows = []
for name, m in models.items():
    base, pert = score(m, test_p), score(m, test_pert_p)
    rows.append({"model": name,
                 "d_f1": round(pert["f1_cash"] - base["f1_cash"], 4),
                 "d_roc_auc": round(pert["roc_auc"] - base["roc_auc"], 4),
                 "d_pr_auc": round(pert["pr_auc"] - base["pr_auc"], 4),
                 "base_f1": round(base["f1_cash"], 4), "base_roc_auc": round(base["roc_auc"], 4)})
stab = pd.DataFrame(rows)
stab["mean_abs_change"] = stab[["d_f1", "d_roc_auc", "d_pr_auc"]].abs().mean(axis=1).round(4)
stab["rank"] = stab["mean_abs_change"].rank(method="min").astype(int)
stab = stab.sort_values("rank")
print(f"perturbation: fare_amount and trip_distance changed by up to +/-{PERT*100:.0f}%")
stab

## 9. Explainability: SHAP on the logistic regression
For a linear model SHAP values can be computed exactly from the coefficients, so this explains the real fitted model, not a stand-in. The other three models are not explained here.

**Fig. T3.2**: the top features by mean absolute SHAP value.

In [ ]:
try:
    import shap
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "shap"])
    import shap

def feature_names(df, n):
    names = [f"f{i}" for i in range(n)]
    try:
        attrs = df.schema["features"].metadata["ml_attr"]["attrs"]
        for group in attrs.values():
            for a in group:
                nm = a["name"]
                if nm.startswith("num_scaled_"):
                    nm = num_cols[int(nm.rsplit("_", 1)[1])].replace("_imp", "")
                names[a["idx"]] = nm
    except Exception as e:
        print("feature names not found in metadata:", e)
    return names

names = feature_names(test_p, n_features)
sample_rows = train_c.sample(False, min(1.0, 4000 / n_train), seed=3).limit(3000).select("features").collect()
X = np.array([r["features"].toArray() for r in sample_rows])

lr_model = models["LogisticRegression"]
coef, intercept = lr_model.coefficients.toArray(), float(lr_model.intercept)
explainer = shap.LinearExplainer((coef, intercept), X)
sv = explainer.shap_values(X[:1000])

mean_abs = np.abs(sv).mean(axis=0)
order = np.argsort(-mean_abs)[:15]
shap_top = [{"feature": names[i], "mean_abs_shap": round(float(mean_abs[i]), 4),
             "direction": "towards cash" if float(np.mean(sv[:, i] * np.sign(X[:1000, i] - X[:, i].mean()))) > 0 else "towards card"}
            for i in order]
shap_table = pd.DataFrame(shap_top)
print("Model explained: LogisticRegression | method: SHAP LinearExplainer | rows explained: 1000")
print(shap_table.head(10).to_string(index=False))

fig, ax = plt.subplots(figsize=(8, 4.5))
ax.barh([s["feature"] for s in shap_top[:10]][::-1], [s["mean_abs_shap"] for s in shap_top[:10]][::-1])
ax.set_xlabel("mean |SHAP value|"); ax.set_title("Top 10 features, logistic regression (SHAP)")
plt.tight_layout(); plt.show()

## 10. Fairness check: pickup zone as a possible proxy
The model sees the pickup and drop-off zone. Zones can stand in for neighbourhoods, and so for the people who live or travel there. This cell compares how often the logistic regression predicts "cash" in each pickup zone with how often cash was really used, for zones with at least 3,000 test trips.

**Fig. T3.3**: predicted against actual cash rate by zone.

In [ ]:
zone = (lr_model.transform(test_p).groupBy("pu_c")
        .agg(F.count("*").alias("trips"),
             F.round(F.avg("prediction") * 100, 2).alias("pred_cash_pct"),
             F.round(F.avg("label") * 100, 2).alias("actual_cash_pct"))
        .filter(F.col("trips") >= 3000).toPandas())
zone["gap_pct_points"] = (zone["pred_cash_pct"] - zone["actual_cash_pct"]).round(2)
zone = zone.sort_values("pred_cash_pct")

lo, hi = zone.iloc[0], zone.iloc[-1]
print(f"zones compared: {len(zone)}")
print(f"lowest predicted cash rate : zone {lo['pu_c']}: {lo['pred_cash_pct']}% (actual {lo['actual_cash_pct']}%)")
print(f"highest predicted cash rate: zone {hi['pu_c']}: {hi['pred_cash_pct']}% (actual {hi['actual_cash_pct']}%)")
print(f"largest over-prediction gap: {zone['gap_pct_points'].max()} points; largest under-prediction: {zone['gap_pct_points'].min()} points")
zone.tail(5)

## 11. results/task3.json

In [ ]:
fairness = {"zones_compared": int(len(zone)), "min_pred_cash_pct": float(lo["pred_cash_pct"]),
            "max_pred_cash_pct": float(hi["pred_cash_pct"]),
            "max_overprediction_gap_points": float(zone["gap_pct_points"].max()),
            "max_underprediction_gap_points": float(zone["gap_pct_points"].min())}
task3 = {
    "student_name": STUDENT_NAME, "sid": SID, "pool_reference": POOL_REF,
    "caching_applied": True, "persist_level": "MEMORY_AND_DISK",
    "optimisations": [opt_cache, opt_shuffle],
    "spark_ui_top_stages": (stage_table.head(5).drop(columns="attempt").to_dict("records") if len(stage_table) else []),
    "stability": {"perturbation": f"fare_amount and trip_distance +/-{PERT*100:.0f}% uniform random",
                  "models": stab.to_dict("records")},
    "explainability": {"method": "SHAP LinearExplainer", "model_explained": "LogisticRegression",
                       "rows_explained": 1000, "top_features": shap_top},
    "bias_risk": {"name": "pickup and drop-off zone as a proxy for neighbourhood and rider group",
                  "check": fairness},
    "cluster_config": CLUSTER, "train_fraction": TRAIN_FRACTION, "test_fraction": TEST_FRACTION,
}
with open(os.path.join(RES_DIR, "task3.json"), "w") as fh:
    json.dump(task3, fh, indent=2)
print("saved results/task3.json")